# Heft 1: Das Zeitreihen-Inventar

Was liegt im System vor? Dieses Heft zeigt das Inventar des Retrieval-Servers als Tabelle, eine gewählte Reihe als
Kurve, ihre Abdeckung je Tag und ihre Lücken, und den Strompreis des Jahres 2025 mit dem Wechsel der Auflösung.

**Regeln dieses Hefts.** Es liest nur; es schreibt nichts in den Broker und nichts in das System. Lücken bleiben
Lücken. Jede Tabelle trägt die Herkunft einer Reihe: gemessen, gemeldet, geschätzt, simuliert.

Zwei Wege zu den Daten: **A** über den Retrieval-Server (Normalfall), **B** direkt aus dem Topic über den Broker,
falls A nicht erreichbar ist. Das Heft wählt selbst und sagt, welchen Weg es nimmt.

Dies ist eine Kopie, die beim Start des Notebooks erneuert wird. Wer damit weiterarbeiten will, speichert sie unter
einem eigenen Namen.

In [ ]:
import os
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from opentsx.client import RetrievalClient, TopicReader
from opentsx.model.pandas_io import episodes_to_frame

RETRIEVAL = os.environ.get("OPENTSX_RETRIEVAL", "http://opentsx-retrieval.opentsx.svc:8080")
BOOTSTRAP = os.environ.get("KAFKA_BOOTSTRAP")
DOMAIN = os.environ.get("OPENTSX_DOMAIN", "wind")
TOPIC = f"tsx.{DOMAIN}.episodes.v2"


def reachable(url: str) -> bool:
    try:
        with urllib.request.urlopen(url + "/v1/health", timeout=5) as r:
            return r.status == 200
    except OSError:
        return False


WAY = "A" if reachable(RETRIEVAL) else ("B" if BOOTSTRAP else None)
print({"A": f"Weg A: Retrieval-Server {RETRIEVAL}",
       "B": f"Weg B: direkt aus {TOPIC} über {BOOTSTRAP} (der Retrieval-Server ist nicht erreichbar)",
       None: "Weder der Retrieval-Server noch ein Broker ist erreichbar: OPENTSX_RETRIEVAL oder KAFKA_BOOTSTRAP setzen."}[WAY])
client = RetrievalClient(RETRIEVAL)
reader = TopicReader(BOOTSTRAP) if BOOTSTRAP else None

## 1. Das Inventar

Eine Zeile je Reihe: was sie misst, in welcher Einheit, von wann bis wann, wie viele Werte, wie viele fehlen, und
woher sie kommt. Über Weg B entsteht die Tabelle aus den Episoden des Topics; das dauert länger und belastet den
Broker, deshalb liest der Leser höchstens so viele Datensätze, wie `MAX_RECORDS` erlaubt.

In [ ]:
MAX_RECORDS = 20_000


def inventory_from_topic() -> pd.DataFrame:
    eps = reader.episodes(TOPIC, max_records=MAX_RECORDS)
    rows = {}
    for e in eps:
        s = e["series"]
        r = rows.setdefault(s["seriesId"], {"series_id": s["seriesId"], "metric": s["metric"], "unit": s.get("unit"),
                                            "from": e["tStart"], "to": e["tEnd"], "samples": 0,
                                            "origin": (e.get("labels") or {}).get("origin"),
                                            **{f"tag.{k}": v for k, v in (s.get("tags") or {}).items()}})
        r["from"], r["to"] = min(r["from"], e["tStart"]), max(r["to"], e["tEnd"])
        r["samples"] += e["count"]
    df = pd.DataFrame(rows.values())
    for c in ("from", "to"):
        df[c] = pd.to_datetime(df[c], unit="us", utc=True)
    return df.sort_values(["metric", "series_id"]).reset_index(drop=True)


inv = client.inventory() if WAY == "A" else inventory_from_topic()
print(f"{len(inv)} Reihen")
inv.groupby(["metric", "origin"], dropna=False).agg(reihen=("series_id", "count"), werte=("samples", "sum"),
                                                    von=("from", "min"), bis=("to", "max"))

## 2. Was gemessen und was gemeldet ist

Die Reihen des Jahres 2025: das Wetter der DWD-Station Greifswalder Oie (gemessen) und der Day-Ahead-Preis
(gemeldet). Die Reihen der Turbinen stammen aus dem Simulator und heißen deshalb simuliert.

In [ ]:
cols = ["metric", "unit", "from", "to", "samples", "missing", "origin"] + [c for c in inv.columns if c.startswith("tag.")]
real = inv[inv["origin"].isin(["MEASURED", "REPORTED"])]
real[[c for c in cols if c in real.columns]].dropna(axis=1, how="all")

## 3. Eine Reihe als Kurve

Gewählt ist der Wind der Station in 10 Minuten. Andere Reihen: `METRIC` und `TAGS` ändern, etwa
`weather.air_temperature` oder `market.price.day_ahead` mit `{"area": "DE-LU"}`.

In [ ]:
METRIC = "weather.wind_speed_10min"
TAGS = {"station": "01759"}
START, END = "2025-01-01T00:00:00Z", "2026-01-01T00:00:00Z"

if WAY == "A":
    frame = client.frame(metric=METRIC, tags=TAGS, start=START, end=END)
else:
    frame = episodes_to_frame(reader.episodes(TOPIC, metric=METRIC, tags=TAGS, start=START, end=END, max_records=MAX_RECORDS))
series = frame.set_index("ts")["value"].sort_index()
print(f"{METRIC} {TAGS}: {len(series)} Werte, {series.index.min()} bis {series.index.max()}")

fig, ax = plt.subplots(figsize=(13, 3.5))
series.plot(ax=ax, lw=0.3, alpha=0.5, label="10 Minuten")
series.resample("1D").mean().plot(ax=ax, lw=1.4, label="Tagesmittel")
ax.set_ylabel(inv.loc[inv["metric"] == METRIC, "unit"].iloc[0] if (inv["metric"] == METRIC).any() else "")
ax.set_title(f"{METRIC} {TAGS}")
ax.legend()
plt.show()

## 4. Abdeckung und Lücken

Je Tag der Anteil der erwarteten Werte, der vorliegt. Eine Lücke wird nicht gefüllt; sie steht in der Liste darunter.

In [ ]:
step = series.index.to_series().diff().mode().iloc[0]
expected_per_day = pd.Timedelta("1D") / step
per_day = series.resample("1D").count() / expected_per_day

grid = per_day.to_frame("anteil")
grid["monat"] = grid.index.month
grid["tag"] = grid.index.day
heat = grid.pivot(index="monat", columns="tag", values="anteil")
fig, ax = plt.subplots(figsize=(13, 3.5))
im = ax.imshow(heat, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
ax.set_xlabel("Tag")
ax.set_ylabel("Monat")
ax.set_xticks(range(0, 31, 5), [str(d) for d in range(1, 32, 5)])
ax.set_yticks(range(len(heat.index)), [str(m) for m in heat.index])
fig.colorbar(im, ax=ax, label="Anteil der erwarteten Werte")
ax.set_title(f"Abdeckung je Tag, Takt {step}")
plt.show()

full = pd.date_range(series.index.min(), series.index.max(), freq=step)
missing = full.difference(series.dropna().index)
runs = []
if len(missing):
    start = prev = missing[0]
    for t in missing[1:]:
        if t - prev != step:
            runs.append((start, prev + step))
            start = t
        prev = t
    runs.append((start, prev + step))
gaps = pd.DataFrame(runs, columns=["von", "bis"])
gaps["dauer"] = gaps["bis"] - gaps["von"]
print(f"{len(missing)} fehlende Werte in {len(gaps)} Lücken")
gaps.sort_values("dauer", ascending=False).head(10)

## 5. Der Strompreis 2025

Der Day-Ahead-Preis ist bis zum Liefertag 2025-09-30 stündlich und danach viertelstündlich. Ein Stundenwert wird
durch Wiederholung keine Viertelstundenmessung; die Kurve zeigt beide Teile mit ihrer eigenen Auflösung.

In [ ]:
PRICE = {"metric": "market.price.day_ahead", "tags": {"area": "DE-LU"}}
if WAY == "A":
    price = client.frame(**PRICE, start=START, end=END)
else:
    price = episodes_to_frame(reader.episodes(TOPIC, **PRICE, start=START, end=END, max_records=MAX_RECORDS))
price = price.set_index("ts")["value"].sort_index()
switch = pd.Timestamp("2025-09-30T22:00:00Z")
print("Auflösung vor dem Wechsel:", price[:switch].index.to_series().diff().mode().iloc[0],
      "| danach:", price[switch:].index.to_series().diff().mode().iloc[0])
print(f"negative Preise: {(price < 0).sum()} Intervalle, niedrigster Preis {price.min():.2f} EUR/MWh")

fig, ax = plt.subplots(figsize=(13, 3.5))
price.plot(ax=ax, lw=0.4)
ax.axvline(switch, color="grey", ls="--", lw=1)
ax.text(switch, ax.get_ylim()[1] * 0.9, "  ab hier viertelstündlich", color="grey")
ax.axhline(0, color="black", lw=0.6)
ax.set_ylabel("EUR/MWh")
ax.set_title("Day-Ahead-Preis DE-LU 2025 (SMARD, gemeldet)")
plt.show()

## Wie es weitergeht

* Heft 2 legt ein Signal der Wasserstoffkette neben seine Erwartung (`client.signals("h2-chain")`).
* Heft 3 prüft eine eigene Datei gegen ein Signal.
* Heft 4 rechnet den Zwilling (`client.scenario(...)`, `client.run(...)`).

Quellen: Deutscher Wetterdienst, CC BY 4.0; Bundesnetzagentur | SMARD.de, CC BY 4.0.